# ImFusion CT 2D/3D Registration

This notebook demonstrates biplanar 2D/3D registration workflow. It simulates projections, perturbs the cone-beam geometry, and runs registration.

## Overview
This tutorial demonstrates a minimal 2D/3D registration workflow:
- Simulate a biplanar acquisition
- Perturb the geometry
- Run 2D/3D registration to recover pose offsets

### Prerequisites
- A working `imfusion-sdk` and `imfusion-sdk-computed_tomography` installation with a valid license

## Setup

### Setup python path
This step can be skipped if the package got installed from PyPI.

In [1]:
# Setup Python path for ImFusion bindings
import sys
import os

# Add the build directory to Python path (adjust if needed)
build_lib_path = '/Users/wieczorek/Desktop/Dev/imfusionsuite/cmake-build-release/lib'
if build_lib_path not in sys.path:
    sys.path.insert(0, build_lib_path)

### Import the modules
We need to import `imfusion` and `imfusion.computed_tomography`.

In [2]:
try:
    import imfusion
    import imfusion.computed_tomography as ct
except ImportError as e:
    raise ImportError("Failed to import ImFusion CT bindings. Make sure the CT plugin is built and the path is correct.") from e


### Data setup
Unzips demo data if needed and available.

In [3]:
import os, sys
sdk_path = os.path.abspath(os.path.join(os.getcwd(), '../imfusion-sdk'))
if sdk_path not in sys.path:
    sys.path.insert(0, sdk_path)

try:
    from demo_utils import unzip_folder
    zip_path = os.path.join('..', 'imfusion-sdk', 'data', 'pet-ct-rtstruct.zip')
    data_dir = os.path.join('..', 'imfusion-sdk', 'data', 'pet-ct-rtstruct')
    if os.path.exists(zip_path) and not os.path.isdir(data_dir):
        print('Unzipping demo data...')
        unzip_folder(zip_path)
    else:
        print('Demo data is present or archive not found. Skipping unzip.')
except Exception as e:
    print(f"Data setup skipped ({e}).")


Demo data is present or archive not found. Skipping unzip.


## Steps in this notebook
1) Load a CT volume and simulate two biplanar shots
2) Introduce known offsets in legacy `ConeBeamGeometry`
3) Run registration and verify the recovered pose

### Load a CT volume and prepare a biplanar acquisition

In [4]:
volume = imfusion.load("../imfusion-sdk/data/pet-ct-rtstruct/ct.imf")[0]
mat = volume.matrix()
mat[0:3, 3] = [0.0, 0.0, 0.0]
volume.set_matrix(mat)

simulator = ct.ConeBeamSimulation(
    volume[0],
    geometry_preset=ct.GeometryPreset.BIPLANAR_SHOT,
    proj_type=ct.ProjectionType.LOG_CONVERTED_ATTENUATION,
    width=1024,
    height=1024,
    frames=2,
    add_poisson_noise=False,
)
projections = simulator()

imfusion.show([projections, volume])


### Perturb legacy geometry to create a registration task

In [5]:
metadata = ct.ConeBeamMetadata.get(projections)
cone_beam_geometry = metadata.geometry()  # legacy geometry
cone_beam_geometry.recon_rot_x += 4.0
cone_beam_geometry.recon_rot_y += 3.0
cone_beam_geometry.angle_start += -6.0
cone_beam_geometry.recon_offset_x += 2.0
cone_beam_geometry.recon_offset_y += -5.0
cone_beam_geometry.recon_offset_z += 7.0

projections_before = projections.clone()


### Run 2D/3D registration

In [7]:
registrator = ct.XRay2D3DRegistration(projections, volume, initialization_mode=ct.InitializationMode.NOOP)
registrator()

print(f"Final translation ({cone_beam_geometry.recon_offset_x}, {cone_beam_geometry.recon_offset_y}, {cone_beam_geometry.recon_offset_z})")
print(f"Final rotation ({cone_beam_geometry.recon_rot_x}, {cone_beam_geometry.recon_rot_y}, {cone_beam_geometry.angle_start})")
imfusion.show([volume, projections_before, projections])

Final translation (0.018709704720393103, 0.0005836458918606533, 0.014890731235423836)
Final rotation (-0.009396631917577338, 0.0025852457406440894, 180.00732913631586)
